# K1: TF-IDF + logistička regresija (McDonald's)
**Tip:** profesorkin primer zadatka (Blok 2). Reprezentacija: VSM / TF-IDF. Klasifikator: logistička regresija.

### Zadatak
U fajlu `McDonald_s_Reviews.csv` su komentari korisnika McDonald's restorana (kolone `reviewer_id`, `store_address`, `rating_count`, `review`, `rating`).
- Od kolone `rating` napraviti varijablu `sentiment` (1 i 2 negativan, 3 neutralan, 4 i 5 pozitivan). To je izlazna varijabla.
- Transformisati kolonu `review` primenom **Vector Space Modela i TF-IDF** metrike. Proceniti da li još neku varijablu treba uključiti i obrazložiti.
- Kreirati prvi model primenom **logističke regresije** sa podrazumevanim hiperparametrima.
- Izračunati preciznost, odziv i F1 po klasama i ukupno. Interpretirati u kontekstu zadatka.
- Kreirati drugi model podešavanjem hiperparametara: komentar šta bi sve podešavao i sa kojim opsezima, pa podesiti dva (jedan za transformaciju teksta, jedan za klasifikator) kroz 5-fold kros-validaciju.
- Predikcije i metrike za podešeni model.
- Uporediti i prokomentarisati osnovni i podešeni model.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import RobustScaler, OneHotEncoder, MinMaxScaler, FunctionTransformer
from sklearn import metrics
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, balanced_accuracy_score
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline   # imblearn Pipeline radi i bez SMOTE-a, pa moze da ostane uvek

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.dummy import DummyClassifier

import matplotlib.pyplot as plt
import seaborn as sb

from pathlib import Path
from IPython.display import display

import spacy

RAND_STATE = 1
DATA_DIR = Path.cwd().parent / 'data'

### Šta je urađeno
- Učitani podaci, izbačena 3 komentara bez teksta.
- Napravljena izlazna varijabla `sentiment` (pozitivan oko 48%, negativan 37%, neutralan 15%).
- Pregledane ostale kolone: `rating_count` (kdeplot), `store_address` (broj vrednosti).
- Odluka: ulaz je samo `review` (`rating_count` ima sličnu raspodelu po klasama, `store_address` ima 37 vrednosti, `reviewer_id` je identifikator, `rating` je izvor izlaza).
- Izlaz kodiran sa `pd.factorize`, podela 80:20 sa `stratify`.

In [ ]:
data = pd.read_csv(DATA_DIR / 'McDonald_s_Reviews.csv')
data.info()

In [ ]:
data = data.dropna(subset=['review']).reset_index(drop=True)
print(data.isna().sum())
print(data['rating'].value_counts())

In [ ]:
def rating_to_sentiment(rating: int):
    if rating < 3:
        return 'negativan'
    return 'neutralan' if rating == 3 else 'pozitivan'

data['sentiment'] = data['rating'].apply(rating_to_sentiment)
print(data['sentiment'].value_counts(normalize=True))

In [ ]:
plt.figure(figsize=(10, 6))
sb.kdeplot(data=data, x='rating_count', hue='sentiment',
           fill=True, common_norm=False, alpha=0.5, linewidth=2)
plt.title('Distribucija broja komentara za razlicite kategorije sentimenta')
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

print('Broj razlicitih adresa:', data['store_address'].nunique())

In [ ]:
data_sub = data[['review', 'sentiment']]

X = data_sub['review']
y = data_sub['sentiment']
y_encodings, y_levels = pd.factorize(y)
print(list(y_levels))

X_train, X_test, y_train, y_test = train_test_split(X, y_encodings,
                                                    test_size=0.2,
                                                    random_state=RAND_STATE,
                                                    stratify=y_encodings)
print(X_train.shape, X_test.shape)

---
# ▶ OVDE KREĆEŠ

Sve iznad je gotovo. Od ove ćelije pa nadalje kucaš sam: kod, komentare u markdown ćelijama i tumačenja.

**Tvoji koraci:**

1. `nlp_pipe = spacy.load('en_core_web_sm', disable=['parser','ner'])` i funkcija `spacy_srediTekst` (zadrži `not`)
2. `TfidfVectorizer(tokenizer=spacy_srediTekst, token_pattern=None, ...)`
3. `LogisticRegression(max_iter=2000)` i `Pipeline` (vectorizer, classifier), pa `fit`
4. Matrica konfuzije, `classification_report`, tumačenje metrika
5. Komentar o hiperparametrima, pa `GridSearchCV` (jedan za TF-IDF + `C`, `cv=5`, `scoring='f1_macro'`)
6. Podešeni model: `best_params_`, `best_score_`, metrike na testu
7. Poređenje sa `get_model_metrics` i komentar

---
## Dati kod
Ovo dobijaš uz zadatak. Pokreni ćeliju kad stigneš do poređenja modela.

In [ ]:
# DATI KOD (dobija se na kolokvijumu): poredjenje modela
def get_model_metrics(y_true: np.ndarray, y_pred: np.ndarray, model_name: str):
    report = classification_report(y_true, y_pred, output_dict=True)
    macro_avg = report['macro avg']
    weighted_avg = report['weighted avg']
    return {
        'Model': model_name,
        'Macro Precision': macro_avg['precision'],
        'Macro Recall': macro_avg['recall'],
        'Macro F1-Score': macro_avg['f1-score'],
        'Weighted Precision': weighted_avg['precision'],
        'Weighted Recall': weighted_avg['recall'],
        'Weighted F1-Score': weighted_avg['f1-score'],
        'Balanced Accuracy': balanced_accuracy_score(y_true, y_pred)
    }